# DSPy 101 — Replacing Prompts with Signatures

**Week 6 | Notebook 1 of 12**

**What you'll learn:**
- LM configuration (OpenAI + Ollama)
- Your first Signature — QuestionAnswering
- dspy.Predict — basic prediction
- dspy.ChainOfThought — adding rationale
- dspy.ProgramOfThought — math problems with code execution
- Side-by-side output comparison across module types
- Inline vs class-based signature syntax

**Runtime:** ~20 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/01_signatures_modules.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/01_signatures_modules.ipynb
Task:      Signatures and core modules
Calls:     ~15

With GPT-4o:       $0.15 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup & LM Configuration

In [2]:
import dspy

from src.config import get_dspy_lm, print_config

print_config()

# Configure DSPy with our unified LM
lm = get_dspy_lm()
dspy.configure(lm=lm)

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      azureopenai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.7-flash
  groq model:      openai/gpt-oss-120b
  azureopenai model: gpt-4o-mini
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: azure/gpt-4o-mini


## 2. Your First Signature — QuestionAnswering

In [3]:
# Class-based signature (recommended for production)
class QuestionAnswering(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="A concise, factual answer")


# Inline signature (quick prototyping)
# inline_sig = "question -> answer"

print("✅ Signature defined")
print(f"  Input: {QuestionAnswering.fields.keys()}")

✅ Signature defined
  Input: dict_keys(['question', 'answer'])


## 3. dspy.Predict — Basic Prediction

In [4]:
# Basic prediction module
predictor = dspy.Predict(QuestionAnswering)

result = predictor(question="What is the capital of India?")
print("Question: What is the capital of India?")
print(f"Answer: {result.answer}")

Question: What is the capital of India?
Answer: The capital of India is New Delhi.


## 4. dspy.ChainOfThought — Adding Rationale

In [5]:
# ChainOfThought automatically adds a 'reasoning' field (dspy 3.x renamed 'rationale')
cot = dspy.ChainOfThought(QuestionAnswering)

result = cot(question="If a train travels 60 km/h for 2.5 hours, how far does it go?")

print("Rationale:")
print(result.reasoning)
print(f"\nAnswer: {result.answer}")

Rationale:
To find the distance traveled by the train, we can use the formula: distance = speed × time. The train travels at a speed of 60 km/h for 2.5 hours. Therefore, the distance covered is 60 km/h × 2.5 h = 150 km.

Answer: 150 km


## 5. dspy.ProgramOfThought — Math with Code Execution

In [6]:
# ProgramOfThought generates and executes Python code
pot = dspy.ProgramOfThought(QuestionAnswering)

result = pot(question="What is the average of 45, 67, 89, and 23?")

print(f"Answer: {result.answer}")
print("\nThis answer was computed by generated Python code, not guessed by the LLM.")

Answer: The average of 45, 67, 89, and 23 is 56.

This answer was computed by generated Python code, not guessed by the LLM.


In [7]:
import dspy
from src.config import get_dspy_lm
# Configure DSPy using your project's .env settings
class QuestionAnswering(dspy.Signature):
    """Answer a question using generated Python code."""
    question = dspy.InputField()
    answer = dspy.OutputField()
# Generate and execute Python code
pot = dspy.ProgramOfThought(QuestionAnswering)
result = pot(
    question="What is the average of 45, 67, 89, and 23?"
)
print("Final result:")
print(result)
print(f"\nAnswer: {result.answer}")
print("\nGenerated code and execution history:")
dspy.inspect_history(n=5)

Final result:
Prediction(
    reasoning='The code calculates the average of the given numbers by first summing them up with `sum(numbers)` which results in 224. Then, it divides the total by the number of elements in the list, which is 4, using `len(numbers)`. The final average is calculated as 224 / 4 = 56, which matches the output provided.',
    answer='The average of 45, 67, 89, and 23 is 56.'
)

Answer: The average of 45, 67, 89, and 23 is 56.

Generated code and execution history:




[2026-09-22T08:12:41.611763]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): A concise, factual answer
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer questions with short factual responses

## 6. Side-by-Side Output Comparison

In [8]:
question = "Explain transformer attention in one sentence."

modules = {
    "Predict": dspy.Predict(QuestionAnswering),
    "ChainOfThought": dspy.ChainOfThought(QuestionAnswering),
    "ProgramOfThought": dspy.ProgramOfThought(QuestionAnswering),
}

print(f"Question: {question}\n")
for name, module in modules.items():
    result = module(question=question)
    print(f"--- {name} ---")
    if hasattr(result, "reasoning"):
        print(f"Reasoning: {result.reasoning[:100]}...")
    print(f"Answer: {result.answer}\n")

Question: Explain transformer attention in one sentence.

--- Predict ---
Answer: Transformer attention is a mechanism that allows the model to weigh the significance of different words in a sequence when generating a representation of a specific word, thus enabling better context understanding and relationships between words.

--- ChainOfThought ---
Reasoning: Transformer attention allows a model to weigh the importance of different tokens in a sequence when ...
Answer: Transformer attention enables a model to focus on relevant parts of the input sequence by assigning different importance to each token based on their contextual relationships.

--- ProgramOfThought ---
Reasoning: The output from the generated code accurately encapsulates the concept of transformer attention by h...
Answer: Transformer attention allows the model to weigh the importance of different words in a sequence relative to each other, facilitating effective context understanding during tasks such as translation a

In [9]:
# Class-based (recommended)
class SentimentClassification(dspy.Signature):
    """Classify the sentiment of the given text."""

    text: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")
    confidence: float = dspy.OutputField(desc="confidence score between 0 and 1")


classifier = dspy.Predict(SentimentClassification)
result = classifier(text="This product is amazing!")
print(f"Class-based: {result.sentiment} (confidence: {result.confidence})")


#------------------------------------------------
# Inline (quick prototyping)
inline_classifier = dspy.Predict("text -> sentiment, confidence")
result2 = inline_classifier(text="This product is amazing!")
print(f"Inline: {result2.sentiment} (confidence: {result2.confidence})")

Class-based: positive (confidence: 0.95)
Inline: positive (confidence: high)
